# Mini RAG Chatbot

## Objective

Build a Retrieval-Augmented Generation (RAG) chatbot that can answer questions based on information contained in a PDF document.

## RAG Pipeline

PDF → Text Extraction → Chunking → Embeddings → FAISS Vector Store → Retrieval → LLM → Answer


## 1. Import Libraries

In [2]:
import os
import numpy as np

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

## 2. Load PDF Document

We will read the PDF using PyPDF and extract the text from each page.

In [3]:
pdf_path = "sample_data/sample.pdf"

reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

Number of pages: 23


## 3. Extract Text from PDF

In [4]:
text = ""

for page in reader.pages:
    page_text = page.extract_text()
    
    if page_text:
        text += page_text + "\n"

print("Total characters:", len(text))

Total characters: 55176


In [5]:
print(text[:3000])

 1 
CHAPTER 1 
INTRODUCTION 
The post-Gupta period, which spanned from circa sixth to the twelfth century a period of 
six centuries, is denoted as early medieval period in the realm of Indian  history, marking 
a transition to medieval times.  Early medieval is used as an historical phase and marked 
off from other historical phases such as  ancient, medieval, and modern. However, the 
notion of these phases being rigidly defined and differentiated is a rela tively recent 
development. What i s more recent is the critical examination of using these terms as 
mere substitute for hindu, muslim and british periods.1  
 The term "early medieval" depicts both a specific chronological period and a set 
of transform ative processes that define this phase. To understand this phase, it i s 
essential to reassess current historical perspectives on the early medieval period and how 
the transition to this phase has been interpreted. By f raming the early medieval period as 
a transitional phase 

## 4. Text Cleaning

In [6]:
text = text.replace("\n", " ")
text = " ".join(text.split())

print(text[:2000])

1 CHAPTER 1 INTRODUCTION The post-Gupta period, which spanned from circa sixth to the twelfth century a period of six centuries, is denoted as early medieval period in the realm of Indian history, marking a transition to medieval times. Early medieval is used as an historical phase and marked off from other historical phases such as ancient, medieval, and modern. However, the notion of these phases being rigidly defined and differentiated is a rela tively recent development. What i s more recent is the critical examination of using these terms as mere substitute for hindu, muslim and british periods.1 The term "early medieval" depicts both a specific chronological period and a set of transform ative processes that define this phase. To understand this phase, it i s essential to reassess current historical perspectives on the early medieval period and how the transition to this phase has been interpreted. By f raming the early medieval period as a transitional phase to the medieval era,

## 5. Text Chunking

Large documents are divided into smaller chunks so that we can retrieve only the relevant portions of the document when a user asks a question.

We use overlapping chunks so that important context is not lost between two consecutive chunks.

In [11]:
chunk_size = 500
overlap = 50

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    
    chunk = text[start:end]
    
    if chunk.strip():
        chunks.append(chunk)
    
    start += chunk_size - overlap

print("Number of chunks:", len(chunks))

Number of chunks: 120


In [12]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk)


--- Chunk 1 ---
1 CHAPTER 1 INTRODUCTION The post-Gupta period, which spanned from circa sixth to the twelfth century a period of six centuries, is denoted as early medieval period in the realm of Indian history, marking a transition to medieval times. Early medieval is used as an historical phase and marked off from other historical phases such as ancient, medieval, and modern. However, the notion of these phases being rigidly defined and differentiated is a rela tively recent development. What i s more recent

--- Chunk 2 ---
la tively recent development. What i s more recent is the critical examination of using these terms as mere substitute for hindu, muslim and british periods.1 The term "early medieval" depicts both a specific chronological period and a set of transform ative processes that define this phase. To understand this phase, it i s essential to reassess current historical perspectives on the early medieval period and how the transition to this phase has been interprete

## 6. Generate Embeddings

Each text chunk is converted into a numerical vector called an embedding.

Embeddings represent the semantic meaning of text. Texts with similar meanings should have similar vector representations.

In this project, we use Google's Gemini embedding model.

In [22]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("API key loaded:", bool(os.getenv("GEMINI_API_KEY")))

API key loaded: True


In [23]:
def create_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    
    return response.embeddings[0].values

In [24]:
test_embedding = create_embedding(chunks[0])

print("Embedding dimensions:", len(test_embedding))

Embedding dimensions: 3072


In [25]:
embeddings = []

for chunk in chunks:
    embedding = create_embedding(chunk)
    embeddings.append(embedding)

print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", len(embeddings[0]))

Number of embeddings: 120
Embedding dimensions: 3072


## 7. Create FAISS Vector Store

FAISS (Facebook AI Similarity Search) is used to store and search the document embeddings efficiently.

Each chunk's embedding is stored as a vector in the FAISS index.

When a user asks a question, the question is also converted into an embedding and FAISS searches for the most similar document chunks.

In [26]:
import faiss
import numpy as np

embedding_matrix = np.array(
    embeddings,
    dtype="float32"
)

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embedding_matrix)

print("Vectors stored in FAISS:", index.ntotal)

Vectors stored in FAISS: 120


## 8. Semantic Search / Retrieval

The user's question is converted into an embedding and compared with the embeddings stored in FAISS.

The most similar chunks are retrieved as the relevant context for answering the question.

This is the Retrieval step in Retrieval-Augmented Generation (RAG).

In [27]:
query = "What is this document about?"

query_embedding = create_embedding(query)

query_embedding = np.array(
    [query_embedding],
    dtype="float32"
)

k = 3

distances, indices = index.search(
    query_embedding,
    k
)

retrieved_chunks = [
    chunks[idx]
    for idx in indices[0]
]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print(chunk)


--- Retrieved Chunk 1 ---
ndia cannot be adequately grasped by merely focusing on dynastic shifts and changes in ruling powers. Instead, historians emphasize the need for a more comprehensive approach that integrates political de velopments with socio-economic and cultural dimensions, including the role of belief systems. 10 By adopting this holistic perspective, scholars aim to capture the complexities and distinctive features of the period. Our work studies the principles, providin g a detailed examination of the early

--- Retrieved Chunk 2 ---
d the managers of these centres. 81 The regulati ons and maintenance of these centres is the focus of this study.82 Various land grants given to these institutions suggest that the financial assistance was provided by various donors. The importance of education was never minimized and was always recogni zed as an important phase in the life of the individual. We have looked into the working of professional groups and how they attained great 

## 9. RAG Prompt & Answer Generation

The retrieved document chunks are combined with the user's question to create a RAG prompt.

The prompt instructs the Gemini language model to answer the question using only the retrieved context from the document.

If the required information is not present in the retrieved context, the model should respond that it does not know the answer based on the provided document.

The prompt is then sent to Google's Gemini language model, which generates the final answer.

This completes the Generation step of the Retrieval-Augmented Generation (RAG) pipeline.

In [29]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
You are a helpful assistant answering questions about a PDF document.

Use only the information provided in the context below.

If the answer is not present in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{query}

Answer:
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

Based on the provided context, the document is an introductory study of the early medieval period in Indian history (the post-Gupta period, spanning from roughly the sixth to the twelfth century). 

Specifically, the document focuses on:
* Examining the period holistically by integrating political developments with socio-economic, cultural, and belief system dimensions.
* Studying the regulations, maintenance, management, and financial support (such as land grants from donors) for educational institutions/centres.
* Investigating the working of professional groups and how they acquired skills and expertise.


## 10. Create the RAG Chatbot Function

The complete RAG pipeline is combined into a single function.

The function takes a user's question, creates its embedding, retrieves the most relevant document chunks from FAISS, builds the RAG prompt, and generates the final answer using Gemini.

In [30]:
def ask_rag(question, k=3):

    # 1. Create embedding for question
    query_embedding = create_embedding(question)

    query_embedding = np.array(
        [query_embedding],
        dtype="float32"
    )

    # 2. Search FAISS
    distances, indices = index.search(
        query_embedding,
        k
    )

    # 3. Retrieve chunks
    retrieved_chunks = [
        chunks[idx]
        for idx in indices[0]
    ]

    # 4. Combine context
    context = "\n\n".join(retrieved_chunks)

    # 5. Create prompt
    prompt = f"""
    You are a helpful assistant answering questions about a PDF document.

    Use only the information provided in the context.

    If the answer is not present in the context, say:
    "I don't know based on the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # 6. Generate answer using Gemini
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

## 11. Test the RAG Chatbot

The RAG chatbot is tested with multiple questions related to the document.

We also test the chatbot with questions whose answers are not present in the document.

This helps verify whether the chatbot correctly uses the retrieved document context instead of generating unsupported answers.

In [31]:
print(ask_rag("What is this document about?"))

Based on the provided document, it is an introduction/study focusing on the early medieval period of Indian history (specifically the post-Gupta period spanning from circa the sixth to the twelfth century). 

Rather than focusing solely on dynastic shifts, the work provides a holistic examination of this era by integrating political, socio-economic, and cultural dimensions (including belief systems). Specifically, it focuses on:
* The regulations, maintenance, and funding (via land grants from donors) of educational institutions/centres.
* The recognized importance of education.
* The working of professional groups and how they attained skills and expertise through various subjects.


In [32]:
print(ask_rag("How were temples contributing to the education?"))

Based on the provided document, temples contributed to education in the following ways:

* They served as learning centers.
* They actively participated in imparting education.
* They paid the teachers who taught the students.


In [33]:
print(ask_rag("How advance and open minded were people in all the aspects that the modern india considers it as a tabbo?"))

I don't know based on the provided document.


In [34]:
print(ask_rag("How adavce the education system was it compared to the mordern India?"))

I don't know based on the provided document.


In [35]:
print(ask_rag("Give me an overview of the education?"))

Based on the provided document, an overview of education includes the following key points:

* **Importance and Intellectual Vitality:** Education was recognized as an important phase in an individual's life and was never minimized. The early medieval education system was marked by exceptional intellectual vitality.
* **Primary Aims and Objectives:** The primary aim of education was the overall development of personality. Objectives included inculcating values such as self-control, character development, self-reliance, social efficacy, and an understanding of duties towards society.
* **Student Preparation and Skill Development:** Students were made self-reliant so they were prepared to face any situation in life. Additionally, professional groups attained great skills and expertise through various subjects.
* **Funding and Dissemination of Knowledge:** Educational institutions received financial assistance from various donors through land grants. Grants of land to Brahmans and monaste

In [38]:
print(ask_rag("Which kingdoms have been mentioned?"))

I don't know based on the provided document.


## 14. Day 1 Conclusion

In Day 1, we implemented the basic Retrieval-Augmented Generation (RAG) pipeline.

The complete workflow is:

PDF → Text Extraction → Text Cleaning → Text Chunking → Gemini Embeddings → FAISS Vector Store → Semantic Retrieval → RAG Prompt → Gemini LLM → Final Answer

The system can now retrieve relevant information from the uploaded PDF and use Gemini to generate answers based on the retrieved context.